# DL_03: BioCause Directed Relation Classification with PubMedBERT
**Causal NLP Reasoning — Phase 2: Deep Learning**

---

## 1. Purpose and Scope
This notebook implements fine-tuning of **PubMedBERT Fulltext** (`microsoft/BiomedNLP-BiomedBERT-base-uncased-abstract-fulltext`) for candidate-conditioned directed causal relation classification on the **BioCause** corpus.

Key requirements:
- Predict directed causal relations ($1 = \text{ARG1 causes ARG2}$, $0 = \text{ARG1 does not cause ARG2 under candidate convention}$).
- Retain explicit argument boundary markers: `[E1]...[/E1]` for Arg1 and `[E2]...[/E2]` for Arg2 in contextual text.
- Standardize on checkpoint `microsoft/BiomedNLP-BiomedBERT-base-uncased-abstract-fulltext`.
- Handle class imbalance using dynamically computed training loss weights ($w_{\text{pos}} = 9,465 / 621 \approx 15.24$). Never hardcode or derive weights from validation/test sets.
- Use conservative GPU defaults: Batch size 8, gradient accumulation 2, FP16 when supported, `eval_strategy='epoch'`.
- Generate predictions for **both** validation and test splits:
  - `data/processed/dl_biocause_validation_predictions.csv` (1,914 rows)
  - `data/processed/dl_biocause_test_predictions.csv` (1,408 rows)
- Include required columns: `candidate_id, article_id, section_id, arg1_id, arg2_id, true_label, dl_prob, dl_pred, split`.
- Preserve raw predicted probabilities; use validation split strictly for decision threshold tuning; never use test labels for tuning.


## 2. Experimental Methodology & Theoretical Background

### Biomedical Domain Pre-training (PubMedBERT Fulltext)
- **PubMedBERT Fulltext** is pretrained *from scratch* on 14M PubMed abstracts and full-text PMC articles with a domain-specific vocabulary.
- It consistently outperforms general-domain BERT on biomedical relation extraction benchmarks.

### Directional Candidate Semantics & Class Imbalance
- **Candidate Generation**: All co-occurring entity pairs within sentences are extracted as directed candidates $(Arg_1, Arg_2)$.
- **Label Semantics**: Label `1` represents directed causation $Arg_1 \to Arg_2$. Label `0` indicates that $Arg_1$ does not cause $Arg_2$ in that candidate configuration. Label `0` includes reversed directional candidates $(Arg_2, Arg_1)$ and non-causal pairs.
- **Dynamic Imbalance Mitigation**: Candidate extraction produces 10,086 training pairs (621 positive, 9,465 negative), giving an actual training ratio of **15.24:1**.
- The loss function incorporates a dynamic positive class weight: $w_{\text{pos}} = \frac{N_{\text{train, neg}}}{N_{\text{train, pos}}} = \frac{9465}{621} \approx 15.2415$.


## 3. Expected Inputs

| Split | Source File | Articles | Gold Relations | Total Candidates | Neg / Pos Ratio |
| :--- | :--- | :---: | :---: | :---: | :---: |
| **Train** | `data/processed/biocause_candidates.csv` (split == 'train') | 12 PMC Articles | 621 | 10,086 | 15.24 : 1 |
| **Validation** | `data/processed/biocause_candidates.csv` (split == 'validation') | 4 PMC Articles | 131 | 1,914 | 13.61 : 1 |
| **Test** | `data/processed/biocause_candidates.csv` (split == 'test') | 4 PMC Articles | 100 | 1,408 | 13.08 : 1 |


## 4. Technical Responsibilities & Implementation Requirements

1. **Portable Root Setup**: Resolve repository path dynamically across environments.
2. **Checkpoint & Tokenizer Configuration**: Standardize on `microsoft/BiomedNLP-BiomedBERT-base-uncased-abstract-fulltext` and register special tokens `['[E1]', '[/E1]', '[E2]', '[/E2]']`.
3. **Embedding Resizing**: Expand model token embeddings to accommodate newly added boundary tokens.
4. **Dynamic Weighted Loss**: Calculate $w_{\text{pos}}$ dynamically from training counts and configure weighted cross-entropy loss.
5. **Conservative Training Configuration**: Batch size 8, gradient accumulation 2, FP16 enabled on CUDA, `eval_strategy='epoch'`.
6. **Validation Prediction & Decision Threshold Tuning**: Predict probabilities on the 4 validation articles (1,914 candidates) and tune decision threshold $\tau^*$ to optimize validation F1.
7. **Held-Out Test Set Prediction**: Predict probabilities on the 1,408 test candidates using frozen $\tau^*$.
8. **Export Dual Prediction Artifacts**: Save both validation and test predictions to `data/processed/` for hybrid integration in `DL_04`.


In [ ]:
# Section 1: Environment Setup & Portable Imports
import sys
import os
from pathlib import Path

def find_repo_root():
    curr = Path(os.getcwd()).resolve()
    for p in [curr] + list(curr.parents):
        if (p / 'AML' / 'causal_extractors.py').exists() and (p / 'DL' / 'dl_utils.py').exists():
            return p
    return curr

ROOT_DIR = find_repo_root()
for p in [ROOT_DIR, ROOT_DIR / 'AML', ROOT_DIR / 'DL']:
    if str(p) not in sys.path:
        sys.path.insert(0, str(p))

import dl_utils
print(f'Repository root located at: {ROOT_DIR}')


In [ ]:
# Section 2: Configurable Hyperparameters & Conservative Resource Defaults
import torch
import pandas as pd
import numpy as np
from transformers import AutoTokenizer, AutoModelForSequenceClassification, Trainer, TrainingArguments

PUBMEDBERT_CHECKPOINT = dl_utils.PUBMEDBERT_CHECKPOINT
assert PUBMEDBERT_CHECKPOINT == 'microsoft/BiomedNLP-BiomedBERT-base-uncased-abstract-fulltext'

# Conservative initial GPU settings
BATCH_SIZE = dl_utils.DEFAULT_BIOCAUSE_BATCH_SIZE  # Conservative default: 8
GRAD_ACCUM_STEPS = dl_utils.DEFAULT_BIOCAUSE_GRAD_ACCUM  # Default: 2
LEARNING_RATE = 2e-5
NUM_EPOCHS = 4
MAX_LENGTH = 384
FP16_ENABLED = torch.cuda.is_available() and not torch.backends.mps.is_available()
EVAL_STRATEGY = 'epoch'  # Modern Transformers API
SAVE_TOTAL_LIMIT = dl_utils.DEFAULT_CHECKPOINT_LIMIT
OUTPUT_DIR = str(ROOT_DIR / 'DL' / 'checkpoints' / 'biocause_pubmedbert')

# TODO: Calculate dynamic class weight strictly from training data (621 pos, 9465 neg)
# pos_weight = dl_utils.calculate_dynamic_class_weight(n_negative=9465, n_positive=621)


In [ ]:
# Section 3: Tokenizer Setup & Special Entity Boundary Markers
# TODO: Load tokenizer and register special tokens: ['[E1]', '[/E1]', '[E2]', '[/E2]']
# tokenizer = AutoTokenizer.from_pretrained(PUBMEDBERT_CHECKPOINT)
# tokenizer.add_special_tokens({'additional_special_tokens': dl_utils.ENTITY_MARKERS})


In [ ]:
# Section 4: Model Instantiation & Embedding Resize
# TODO: Instantiate PubMedBERT for binary classification and resize embeddings
# model = AutoModelForSequenceClassification.from_pretrained(PUBMEDBERT_CHECKPOINT, num_labels=2)
# model.resize_token_embeddings(len(tokenizer))


In [ ]:
# Section 5: Training Pipeline Execution
# TODO: Configure Trainer with dynamic class-weighted loss and conservative resource arguments
# training_args = TrainingArguments(
#     output_dir=OUTPUT_DIR,
#     per_device_train_batch_size=BATCH_SIZE,
#     gradient_accumulation_steps=GRAD_ACCUM_STEPS,
#     eval_strategy=EVAL_STRATEGY,
#     save_total_limit=SAVE_TOTAL_LIMIT,
#     fp16=FP16_ENABLED,
#     metric_for_best_model='eval_f1'
# )


In [ ]:
# Section 6: Validation Predictions & Decision Threshold Tuning
# TODO: Generate raw predicted probabilities for validation candidates (1,914 rows)
# - Sweep decision threshold tau in [0.2, 0.8] strictly on validation set to maximize F1
# - Record optimal threshold tau*
# - Export validation predictions:
#   data/processed/dl_biocause_validation_predictions.csv (1,914 rows)
#   Columns: ['candidate_id', 'article_id', 'section_id', 'arg1_id', 'arg2_id', 'true_label', 'dl_prob', 'dl_pred', 'split']


In [ ]:
# Section 7: Held-Out Test Evaluation & Export
# TODO: Generate predictions on held-out test candidates (1,408 rows) using frozen tau*
# - Never tune tau on test labels!
# - Export test predictions:
#   data/processed/dl_biocause_test_predictions.csv (1,408 rows)
#   Columns: ['candidate_id', 'article_id', 'section_id', 'arg1_id', 'arg2_id', 'true_label', 'dl_prob', 'dl_pred', 'split']


## 5. Expected Outputs & Evaluation Criteria

- Best fine-tuned PubMedBERT checkpoint.
- `data/processed/dl_biocause_validation_predictions.csv`: 1,914 validation candidates with raw probabilities and decisions.
- `data/processed/dl_biocause_test_predictions.csv`: 1,408 test candidates with raw probabilities and decisions.
- **Evaluation Criteria**: Candidate-level Precision, Recall, and F1 on the 4 held-out test publications (100 gold relations, 1,408 candidates).
